# Revisión de Etiquetas en Archivos SAV (ENEMDU)

Este notebook extrae las etiquetas (value_labels) de los archivos .sav de ENEMDU (2021-2025)
y las compara con los mapeos documentados en `enemdu_mappings.py`.

## Objetivo
- Extraer `variable_value_labels` de cada archivo .sav por año
- Identificar variables con etiquetas
- Comparar con mapeos fallback y patrones semánticos en `enemdu_mappings.py`
- Detectar inconsistencias o nuevas etiquetas entre años

In [12]:
#%pip install pyreadstat

In [13]:
import pyreadstat
import pandas as pd
from pathlib import Path
import json
import sys
import re

# Agregar src al path para importar enemdu_mappings
sys.path.insert(0, '../src/processing')
from enemdu_mappings import (
    FALLBACK_AREA, FALLBACK_SEXO, FALLBACK_NIVEL_INSTRUCCION,
    FALLBACK_CONDACT, COLUMN_ALIASES,
    classify_labels, resolve_map, EDU_PATTERNS, CONDACT_PATTERNS
)

# Configurar rutas
BRONZE_DIR = Path('../data/bronze/externas/ENEMDU')
SAV_FILES = sorted(BRONZE_DIR.glob('*.sav'))

print(f"Archivos .sav encontrados: {len(SAV_FILES)}")
for f in SAV_FILES:
    print(f"  - {f.name}")

Archivos .sav encontrados: 5
  - BDDenemdu_personas_2021_anual.sav
  - BDDenemdu_personas_2022_anual.sav
  - BDDenemdu_personas_2023_anual.sav
  - BDDenemdu_personas_2024_anual.sav
  - BDDenemdu_personas_2025_anual.sav


## 1. Extracción de etiquetas por año

In [14]:
# Diccionario para almacenar etiquetas por año
etiquetas_por_ano = {}

for sav_file in SAV_FILES:
    # Extraer año del nombre: BDDenemdu_personas_YYYY_anual.sav
    ano = sav_file.name.split('_')[-2]
    
    print(f"\n{'='*60}")
    print(f"Año: {ano} | Archivo: {sav_file.name}")
    print(f"{'='*60}")
    
    # Leer archivo .sav
    df, meta = pyreadstat.read_sav(str(sav_file))
    
    # Obtener etiquetas de valores
    value_labels = meta.variable_value_labels or {}
    
    print(f"Variables con etiquetas: {len(value_labels)}")
    print(f"Dimensiones del DataFrame: {df.shape}")
    
    # Almacenar para análisis posterior
    etiquetas_por_ano[ano] = {
        'df': df,
        'meta': meta,
        'value_labels': value_labels,
        'columnas': list(df.columns)
    }
    
    # Mostrar primeras variables con etiquetas
    print(f"\nPrimeras 5 variables con etiquetas:")
    for i, (var, labels) in enumerate(list(value_labels.items())[:5]):
        print(f"  {var}: {list(labels.items())[:3]}...")


Año: 2021 | Archivo: BDDenemdu_personas_2021_anual.sav
Variables con etiquetas: 120
Dimensiones del DataFrame: (361790, 151)

Primeras 5 variables con etiquetas:
  area: [(1.0, 'Urbana'), (2.0, 'Rural')]...
  p02: [(1.0, 'Hombre'), (2.0, 'Mujer')]...
  p03: [(98.0, '98 y más'), (99.0, 'No informa')]...
  p04: [(1.0, 'Jefe'), (2.0, 'Cónyuge'), (3.0, 'Hijo o Hija')]...
  p05a: [(1.0, 'IESS, seguro general'), (2.0, 'IESS, seguro general voluntario'), (3.0, 'IESS, seguro  campesino')]...

Año: 2022 | Archivo: BDDenemdu_personas_2022_anual.sav
Variables con etiquetas: 108
Dimensiones del DataFrame: (358096, 139)

Primeras 5 variables con etiquetas:
  area: [(1.0, ' Urbana'), (2.0, ' Rural')]...
  p02: [(1.0, ' Hombre'), (2.0, ' Mujer')]...
  p03: [(98.0, ' 98 y más'), (99.0, ' No informa')]...
  p04: [(1.0, ' Jefe'), (2.0, ' Cónyuge'), (3.0, ' Hijo o Hija')]...
  p05a: [(1.0, ' IESS, seguro general'), (2.0, ' IESS, seguro general voluntario'), (3.0, ' IESS, seguro  campesino')]...

Año: 20

## 2. Análisis por variable: comparar etiquetas con mapeos

In [15]:
# Crear un resumen de variables claves y sus etiquetas por año

# Variables de interés según enemdu_mappings.py
VARIABLES_INTERES = {
    'area': ['area', 'area_geografica'],  # FALLBACK_AREA
    'sexo': ['p02', 'sexo', 'p2'],         # FALLBACK_SEXO
    'nivel_instruccion': ['p10a', 'nivel_instruccion', 'nivelins', 'nnivins'],  # EDU
    'condicion_actividad': ['condact', 'condactn', 'condicion_actividad'],  # CONDACT
}

resumen_variables = {}

for ano, data in sorted(etiquetas_por_ano.items()):
    resumen_variables[ano] = {}
    value_labels = data['value_labels']
    
    for var_grupo, aliases in VARIABLES_INTERES.items():
        # Encontrar cuál alias existe en el archivo
        etiqueta_encontrada = None
        var_encontrada = None
        
        for alias in aliases:
            if alias in value_labels:
                etiqueta_encontrada = value_labels[alias]
                var_encontrada = alias
                break
        
        if etiqueta_encontrada:
            resumen_variables[ano][var_grupo] = {
                'variable': var_encontrada,
                'etiquetas': etiqueta_encontrada
            }

print("RESUMEN: Variables por año")
print(json.dumps(resumen_variables, indent=2, default=str))

RESUMEN: Variables por año
{
  "2021": {
    "area": {
      "variable": "area",
      "etiquetas": {
        "1.0": "Urbana",
        "2.0": "Rural"
      }
    },
    "sexo": {
      "variable": "p02",
      "etiquetas": {
        "1.0": "Hombre",
        "2.0": "Mujer"
      }
    },
    "nivel_instruccion": {
      "variable": "p10a",
      "etiquetas": {
        "1.0": "Ninguno",
        "2.0": "Centro de alfabetizaci\u00f3n",
        "3.0": "Jard\u00edn de infantes",
        "4.0": "Primaria",
        "5.0": "Educaci\u00f3n B\u00e1sica",
        "6.0": "Secundaria",
        "7.0": "Educaci\u00f3n  Media",
        "8.0": "Superior no universitario",
        "9.0": "Superior Universitario",
        "10.0": "Post-grado"
      }
    },
    "condicion_actividad": {
      "variable": "condact",
      "etiquetas": {
        "0.0": "Menores de 15 a\u00f1os",
        "1.0": "Adecuado",
        "2.0": "Subempleo por insuficiencia de tiempo de trabajo",
        "3.0": "Subempleo por insufic

## 3. Comparación: Etiquetas del SAV vs Mapeos en enemdu_mappings.py

In [16]:
# Comparación detallada: SEXO
print("\n" + "="*70)
print("VARIABLE: SEXO (FALLBACK_SEXO)")
print("="*70)
print(f"\nMapeo esperado (fallback): {FALLBACK_SEXO}")
print("\nEtiquetas encontradas en .sav por año:")

for ano, data in sorted(etiquetas_por_ano.items()):
    print(f"\n  {ano}:")
    value_labels = data['value_labels']
    
    for alias in ['p02', 'sexo', 'p2']:
        if alias in value_labels:
            labels = value_labels[alias]
            print(f"    Variable '{alias}': {dict(labels)}")
            break


VARIABLE: SEXO (FALLBACK_SEXO)

Mapeo esperado (fallback): {1: 'Hombre', 2: 'Mujer'}

Etiquetas encontradas en .sav por año:

  2021:
    Variable 'p02': {1.0: 'Hombre', 2.0: 'Mujer'}

  2022:
    Variable 'p02': {1.0: ' Hombre', 2.0: ' Mujer'}

  2023:
    Variable 'p02': {1.0: ' Hombre', 2.0: ' Mujer'}

  2024:
    Variable 'p02': {1.0: ' Hombre', 2.0: ' Mujer'}

  2025:
    Variable 'p02': {1.0: ' Hombre', 2.0: ' Mujer'}


In [17]:
# Comparación detallada: NIVEL DE INSTRUCCIÓN
print("\n" + "="*70)
print("VARIABLE: NIVEL DE INSTRUCCIÓN (EDU_PATTERNS)")
print("="*70)
print(f"\nMapeo esperado (fallback): {FALLBACK_NIVEL_INSTRUCCION}")
print(f"\nPatrones de clasificación (EDU_PATTERNS):")
for cat, pattern in EDU_PATTERNS:
    print(f"  {cat}: {pattern}")

print("\n\nEtiquetas encontradas en .sav por año:")

for ano, data in sorted(etiquetas_por_ano.items()):
    print(f"\n  {ano}:")
    value_labels = data['value_labels']
    
    for alias in ['p10a', 'nivel_instruccion', 'nivelins', 'nnivins']:
        if alias in value_labels:
            labels = value_labels[alias]
            print(f"    Variable '{alias}':")
            for code, text in dict(labels).items():
                print(f"      {code}: {text}")
            break


VARIABLE: NIVEL DE INSTRUCCIÓN (EDU_PATTERNS)

Mapeo esperado (fallback): {1: 'Ninguno', 2: 'Centro de alfabetización', 3: 'Jardín de infantes', 4: 'Primaria', 5: 'Educación Básica', 6: 'Secundaria', 7: 'Educación Media/Bachillerato', 8: 'Superior no universitaria', 9: 'Superior universitaria', 10: 'Post-grado'}

Patrones de clasificación (EDU_PATTERNS):
  Post-grado: post\s*-?\s*grado|maestr|doctor|phd|especializac
  Superior universitaria: superior\s+universitar|universitar
  Superior no universitaria: superior\s+no\s+universitar|tecnolog|tecnic\w*\s+superior
  Educación Media/Bachillerato: media|bachiller
  Secundaria: secundar
  Educación Básica: b[aá]sic
  Primaria: primar
  Jardín de infantes: jard[ií]n|preescolar|inicial
  Centro de alfabetización: alfabetiz
  Ninguno: ningun|sin\s+instrucc


Etiquetas encontradas en .sav por año:

  2021:
    Variable 'p10a':
      1.0: Ninguno
      2.0: Centro de alfabetización
      3.0: Jardín de infantes
      4.0: Primaria
      5.0: Edu

In [18]:
# Comparación detallada: CONDICIÓN DE ACTIVIDAD
print("\n" + "="*70)
print("VARIABLE: CONDICIÓN DE ACTIVIDAD (CONDACT_PATTERNS)")
print("="*70)
print(f"\nMapeo esperado (fallback): {FALLBACK_CONDACT}")
print(f"\nPatrones de clasificación (CONDACT_PATTERNS):")
for cat, pattern in CONDACT_PATTERNS:
    print(f"  {cat}: {pattern}")

print("\n\nEtiquetas encontradas en .sav por año:")

for ano, data in sorted(etiquetas_por_ano.items()):
    print(f"\n  {ano}:")
    value_labels = data['value_labels']
    
    for alias in ['condact', 'condactn', 'condicion_actividad']:
        if alias in value_labels:
            labels = value_labels[alias]
            print(f"    Variable '{alias}':")
            for code, text in dict(labels).items():
                print(f"      {code}: {text}")
            # Aplicar clasificación
            classified = classify_labels(labels, CONDACT_PATTERNS)
            print(f"    Clasificadas: {classified}")
            break


VARIABLE: CONDICIÓN DE ACTIVIDAD (CONDACT_PATTERNS)

Mapeo esperado (fallback): {1: 'Empleo adecuado/pleno', 2: 'Subempleo por insuficiencia de tiempo de trabajo', 3: 'Subempleo por insuficiencia de ingresos', 4: 'Otro empleo no pleno', 5: 'Empleo no remunerado', 6: 'Empleo no clasificado', 7: 'Desempleo abierto', 8: 'Desempleo oculto', 9: 'Población económicamente inactiva (PEI)', 10: 'Menor de 15 años'}

Patrones de clasificación (CONDACT_PATTERNS):
  Empleo adecuado/pleno: adecuad|pleno
  Subempleo por insuficiencia de tiempo de trabajo: subempleo.*tiempo|tiempo.*subempleo
  Subempleo por insuficiencia de ingresos: subempleo.*ingres|ingres.*subempleo
  Otro empleo no pleno: otro\s+empleo|no\s+pleno
  Empleo no remunerado: no\s+remunerad
  Empleo no clasificado: no\s+clasificad
  Desempleo abierto: desempleo\s+abierto
  Desempleo oculto: desempleo\s+oculto
  Población económicamente inactiva (PEI): inactiv|pei
  Menor de 15 años: menor(es)?\s+de\s+1[05]


Etiquetas encontradas en .s

In [19]:
# Comparación detallada: ÁREA GEOGRÁFICA
print("\n" + "="*70)
print("VARIABLE: ÁREA GEOGRÁFICA (FALLBACK_AREA)")
print("="*70)
print(f"\nMapeo esperado (fallback): {FALLBACK_AREA}")
print("\nEtiquetas encontradas en .sav por año:")

for ano, data in sorted(etiquetas_por_ano.items()):
    print(f"\n  {ano}:")
    value_labels = data['value_labels']
    
    for alias in ['area', 'area_geografica']:
        if alias in value_labels:
            labels = value_labels[alias]
            print(f"    Variable '{alias}': {dict(labels)}")
            break


VARIABLE: ÁREA GEOGRÁFICA (FALLBACK_AREA)

Mapeo esperado (fallback): {1: 'Urbana', 2: 'Rural'}

Etiquetas encontradas en .sav por año:

  2021:
    Variable 'area': {1.0: 'Urbana', 2.0: 'Rural'}

  2022:
    Variable 'area': {1.0: ' Urbana', 2.0: ' Rural'}

  2023:
    Variable 'area': {1.0: ' Urbana', 2.0: ' Rural'}

  2024:
    Variable 'area': {1.0: ' Urbana', 2.0: ' Rural'}

  2025:
    Variable 'area': {1.0: ' Urbana', 2.0: ' Rural'}


## 4. Listado completo de variables con etiquetas por año

In [20]:
# Tabla resumen: todas las variables con etiquetas
for ano, data in sorted(etiquetas_por_ano.items()):
    print(f"\n{'='*70}")
    print(f"AÑO {ano} - Variables con etiquetas ({len(data['value_labels'])})")
    print(f"{'='*70}")
    
    value_labels = data['value_labels']
    for var_name in sorted(value_labels.keys()):
        labels = value_labels[var_name]
        n_labels = len(labels)
        print(f"\n  {var_name} ({n_labels} etiquetas):")
        for code, text in dict(labels).items():
            print(f"    {code}: {text}")


AÑO 2021 - Variables con etiquetas (120)

  area (2 etiquetas):
    1.0: Urbana
    2.0: Rural

  ced01a (3 etiquetas):
    1.0: Si
    2.0: No
    3.0: No responde

  cod_inf (25 etiquetas):
    1.0: Persona 1
    2.0: Persona 2
    3.0: Persona 3
    4.0: Persona 4
    5.0: Persona 5
    6.0: Persona 6
    7.0: Persona 7
    8.0: Persona 8
    9.0: Persona 9
    10.0: Persona 10
    11.0: Persona 11
    12.0: Persona 12
    13.0: Persona 13
    14.0: Persona 14
    15.0: Persona 15
    16.0: Persona 16
    17.0: Persona 17
    18.0: Persona 18
    19.0: Persona 19
    20.0: Persona 20
    21.0: Persona 21
    22.0: Persona 22
    23.0: Persona 23
    24.0: Persona 24
    25.0: Persona 25

  condact (10 etiquetas):
    0.0: Menores de 15 años
    1.0: Adecuado
    2.0: Subempleo por insuficiencia de tiempo de trabajo
    3.0: Subempleo por insuficiencia de ingresos
    4.0: Otro empleo inadecuado
    5.0: Empleo no remunerado
    6.0: Empleo no clasificado
    7.0: Desempleo abierto


## 5. Análisis de diferencias entre años

In [21]:
# Identificar variables que existen en algunos años pero no en otros
all_vars = set()
for ano, data in etiquetas_por_ano.items():
    all_vars.update(data['value_labels'].keys())

print("\nVariables que cambian entre años:")
print("="*70)

for var in sorted(all_vars):
    presencia = {}
    for ano in sorted(etiquetas_por_ano.keys()):
        presencia[ano] = var in etiquetas_por_ano[ano]['value_labels']
    
    # Mostrar solo si cambia entre años
    if not all(presencia.values()) or not all(v == presencia['2021'] for v in presencia.values()):
        print(f"\n{var}:")
        for ano, existe in sorted(presencia.items()):
            estado = "✓ Sí" if existe else "✗ No"
            print(f"  {ano}: {estado}")


Variables que cambian entre años:

hogar:
  2021: ✗ No
  2022: ✗ No
  2023: ✗ No
  2024: ✓ Sí
  2025: ✓ Sí

p01:
  2021: ✗ No
  2022: ✗ No
  2023: ✗ No
  2024: ✓ Sí
  2025: ✓ Sí

p081:
  2021: ✗ No
  2022: ✗ No
  2023: ✓ Sí
  2024: ✗ No
  2025: ✗ No

p085:
  2021: ✗ No
  2022: ✗ No
  2023: ✓ Sí
  2024: ✗ No
  2025: ✗ No

p10a:
  2021: ✓ Sí
  2022: ✓ Sí
  2023: ✗ No
  2024: ✓ Sí
  2025: ✓ Sí

p59:
  2021: ✓ Sí
  2022: ✗ No
  2023: ✗ No
  2024: ✗ No
  2025: ✗ No

p60a:
  2021: ✓ Sí
  2022: ✗ No
  2023: ✗ No
  2024: ✗ No
  2025: ✗ No

p60b:
  2021: ✓ Sí
  2022: ✗ No
  2023: ✗ No
  2024: ✗ No
  2025: ✗ No

p60c:
  2021: ✓ Sí
  2022: ✗ No
  2023: ✗ No
  2024: ✗ No
  2025: ✗ No

p60d:
  2021: ✓ Sí
  2022: ✗ No
  2023: ✗ No
  2024: ✗ No
  2025: ✗ No

p60e:
  2021: ✓ Sí
  2022: ✗ No
  2023: ✗ No
  2024: ✗ No
  2025: ✗ No

p60f:
  2021: ✓ Sí
  2022: ✗ No
  2023: ✗ No
  2024: ✗ No
  2025: ✗ No

p60g:
  2021: ✓ Sí
  2022: ✗ No
  2023: ✗ No
  2024: ✗ No
  2025: ✗ No

p60h:
  2021: ✓ Sí
  2022: ✗ 

## 6. Prueba de funciones classify_labels y resolve_map

In [22]:
# Probar resolve_map para diferentes variables y años
print("\nPrueba de resolve_map():")
print("="*70)

test_cases = [
    ('nivel_instruccion', ['p10a', 'nivel_instruccion', 'nivelins', 'nnivins'], EDU_PATTERNS, FALLBACK_NIVEL_INSTRUCCION),
    ('condicion_actividad', ['condact', 'condactn', 'condicion_actividad'], CONDACT_PATTERNS, FALLBACK_CONDACT),
]

for var_name, aliases, patterns, fallback in test_cases:
    print(f"\n\n{var_name.upper()}")
    print("-" * 70)
    
    for ano, data in sorted(etiquetas_por_ano.items()):
        value_labels = data['value_labels']
        
        # Encontrar alias que existe
        labels_dict = None
        alias_usado = None
        for alias in aliases:
            if alias in value_labels:
                labels_dict = value_labels[alias]
                alias_usado = alias
                break
        
        if labels_dict:
            result_map, procedencia = resolve_map(labels_dict, patterns, fallback)
            print(f"\n  {ano} (variable '{alias_usado}'):")
            print(f"    Procedencia: {procedencia}")
            print(f"    Mapeo: {result_map}")


Prueba de resolve_map():


NIVEL_INSTRUCCION
----------------------------------------------------------------------

  2021 (variable 'p10a'):
    Procedencia: diccionario
    Mapeo: {1: 'Ninguno', 2: 'Centro de alfabetización', 3: 'Jardín de infantes', 4: 'Primaria', 5: 'Educación Básica', 6: 'Secundaria', 7: 'Educación Media/Bachillerato', 8: 'Superior universitaria', 9: 'Superior universitaria', 10: 'Post-grado'}

  2022 (variable 'p10a'):
    Procedencia: diccionario
    Mapeo: {1: 'Ninguno', 2: 'Centro de alfabetización', 3: 'Jardín de infantes', 4: 'Primaria', 5: 'Educación Básica', 6: 'Secundaria', 7: 'Educación Media/Bachillerato', 8: 'Superior universitaria', 9: 'Superior universitaria', 10: 'Post-grado'}

  2023 (variable 'nnivins'):
    Procedencia: fallback
    Mapeo: {1: 'Ninguno', 2: 'Centro de alfabetización', 3: 'Jardín de infantes', 4: 'Primaria', 5: 'Educación Básica', 6: 'Secundaria', 7: 'Educación Media/Bachillerato', 8: 'Superior no universitaria', 9: 'Superior un